In [3]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [4]:
# clone your branch directly
!git clone -b Valeria https://github.com/paolito81/MaskArchitectureAnomaly_CourseProject

%cd /content/MaskArchitectureAnomaly_CourseProject/eomt/

%pip install -r requirements.txt

fatal: destination path 'MaskArchitectureAnomaly_CourseProject' already exists and is not an empty directory.
/content/MaskArchitectureAnomaly_CourseProject/eomt


In [5]:
cityscapes_config_path = "/content/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
cityscapes_data_path = "/content/drive/MyDrive/dataset_city_scapes/cityscapes"
cityscapes_ckpt_path = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_cityscapes.bin"

In [6]:
coco_config_path = "/content/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
coco_data_path = "/content/drive/MyDrive/dataset_city_scapes/coco"
coco_ckpt_path = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_coco.bin"

In [7]:
coco_copy_config_path = "/content/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640_coco.yaml"

In [8]:
!git pull
!git branch -a

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1012 bytes | 253.00 KiB/s, done.
From https://github.com/paolito81/MaskArchitectureAnomaly_CourseProject
   2488d8c..d5bb2a4  Valeria    -> origin/Valeria
Updating 2488d8c..d5bb2a4
Fast-forward
 eomt/colab notebooks/COCO_finetuning.ipynb | 98 ++++++++++++------------------
 1 file changed, 39 insertions(+), 59 deletions(-)
* Valeria
  remotes/origin/HEAD -> origin/main
  remotes/origin/Hossein
  remotes/origin/Omar
  remotes/origin/Valeria
  remotes/origin/lora-eomt-test
  remotes/origin/main


In [9]:
import os

os.environ["WANDB_API_KEY"] = (
    "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"
)

In [10]:
coco_copy_config_path_eval = "/content/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640_finetune.yaml"

In [11]:
from datetime import datetime

run_name = datetime.now().strftime("run_%Y%m%d_%H%M%S")
default_root_dir = f"/content/drive/MyDrive/eomt_runs/{run_name}"

In [ ]:
!python3 main.py fit \
   -c {coco_copy_config_path_eval} \
   --trainer.max_epochs 3 \
   --model.load_ckpt_class_head False \
   --trainer.default_root_dir {default_root_dir} \
   --trainer.devices 1 \
   --data.batch_size 2 \
   --data.path {cityscapes_data_path} \
   --model.ckpt_path {coco_ckpt_path} 

2026-05-18 23:07:09.230583: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
INFO:root:Loaded 195 keys
Using 16bit Automatic Mixed Precision (AMP)
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
wandb: Currently logged in as: valeriaintini2001 (valeriaintini2001-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.19.10
wandb: Run data is saved locally in ./wandb/run-20260518_230718-bepcv3ev
wandb: Run `wandb offline` to turn off sy

In [ ]:
import glob

ckpts = sorted(
    glob.glob(f"{default_root_dir}/**/*.ckpt", recursive=True)
)

print("Found checkpoints:")
for c in ckpts:
    print(c)

fine_tuned_coco_ckpt_path = ckpts[-1]

print("\nUsing latest checkpoint:")
print(fine_tuned_coco_ckpt_path)

Found checkpoints:
/content/drive/MyDrive/eomt_runs/epoch=0-step=1487.ckpt

Using latest checkpoint:
/content/drive/MyDrive/eomt_runs/epoch=0-step=1487.ckpt


In [ ]:
!python3 eval_shared_miou.py \
  --config {coco_copy_config_path_eval} \
  --ckpt {fine_tuned_coco_ckpt_path} \
  --cityscapes-path {cityscapes_data_path} \
  --device cuda:0 \
  --batch-size 4 \
  --num-workers 2 \
  --no-masked-attn-enabled \
  --wandb-mode online \
  --src-label-space cityscapes

wandb: Currently logged in as: valeriaintini2001 (valeriaintini2001-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.19.10
wandb: Run data is saved locally in /content/MaskArchitectureAnomaly_CourseProject/eomt/wandb/run-20260518_224253-4qeeeqsp
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run coco_panoptic_eomt_base_640
wandb: ⭐️ View project at https://wandb.ai/valeriaintini2001-politecnico-di-torino/eomt
wandb: 🚀 View run at https://wandb.ai/valeriaintini2001-politecnico-di-torino/eomt/runs/4qeeeqsp
2026-05-18 22:43:01.188926: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/p

In [ ]:
print("Estimated stepping batches:", self.trainer.estimated_stepping_batches)

NameError: name 'self' is not defined